In [4]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

env_path = r"D:\common_credentials\.env"
load_dotenv(dotenv_path=env_path)

True

In [7]:
model = ChatGroq() #mixtral-8x7b-32768

## Runnable lambda:
- ***convert lambda function or any function to the Runnable|chain object***

In [10]:
from langchain_core.runnables import RunnableLambda

In [15]:
def addition(x:int, y:int) -> int:
    return x+y

runnable= RunnableLambda(addition)
runnable

RunnableLambda(addition)

# Runnable Parallel

In [20]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

def add_one(x: int) -> int:
    return x + 1

def mul_two(x: int) -> int:
    return x * 2

def mul_three(x: int) -> int:
    return x * 3

runnable_1 = RunnableLambda(add_one)
runnable_2 = RunnableLambda(mul_two)
runnable_3 = RunnableLambda(mul_three)


# Or equivalently:
sequence = runnable_1 | RunnableParallel(
    {"mul_two": runnable_2, "mul_three": runnable_3}
)

sequence.invoke(1)


{'mul_two': 4, 'mul_three': 6}

In [23]:
sequence.get_graph().print_ascii()

          +---------------+            
          | add_one_input |            
          +---------------+            
                  *                    
                  *                    
                  *                    
             +---------+               
             | add_one |               
             +---------+               
                  *                    
                  *                    
                  *                    
+----------------------------------+   
| Parallel<mul_two,mul_three>Input |   
+----------------------------------+   
            ***        ***             
           *              *            
         **                **          
  +---------+           +-----------+  
  | mul_two |           | mul_three |  
  +---------+           +-----------+  
            ***        ***             
               *      *                
                **  **                 
+-----------------------------------+  


## Runnable parrallel Example 2

In [25]:
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

In [29]:
#Short Notes PRompt
prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

#QnA prompt
prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)


parser= StrOutputParser()
parser

StrOutputParser()

In [36]:
#parallel chain 

parallel_chain= RunnableParallel({
    "notes": prompt1 | model | parser , 
    "quiz" : prompt2 | model | parser
})

#merge chain
merge_chain = prompt3 | model | parser

#main chain 
chain = parallel_chain | merge_chain

In [40]:
text= """"Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

## Invoke the Chain 
result= chain.invoke({"text": text})

In [41]:
result

'Support vector machines (SVMs) are used for classification, regression, and outliers detection. They have several advantages, including effectiveness in high dimensional spaces, still being effective when the number of dimensions is greater than the number of samples, using a subset of training points in the decision function, and versatility with different kernel functions available and the option for custom kernels. However, there are also some disadvantages to using SVMs. Over-fitting can occur if the number of features is greater than the number of samples, careful choice of kernel functions and regularization term is needed. SVMs do not directly provide probability estimates, these are calculated using five-fold cross-validation which can be expensive. For making predictions with sparse data, SVM must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr\\_matrix (sparse) with dtype=float64.\n\nQuiz:\n\n1. What are support ve

In [43]:
Markdown(result)

Support vector machines (SVMs) are used for classification, regression, and outliers detection. They have several advantages, including effectiveness in high dimensional spaces, still being effective when the number of dimensions is greater than the number of samples, using a subset of training points in the decision function, and versatility with different kernel functions available and the option for custom kernels. However, there are also some disadvantages to using SVMs. Over-fitting can occur if the number of features is greater than the number of samples, careful choice of kernel functions and regularization term is needed. SVMs do not directly provide probability estimates, these are calculated using five-fold cross-validation which can be expensive. For making predictions with sparse data, SVM must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr\_matrix (sparse) with dtype=float64.

Quiz:

1. What are support vector machines used for?
Support vector machines are used for classification, regression, and outliers detection.

2. What are some advantages of support vector machines?
Support vector machines are effective in high dimensional spaces, still effective when the number of dimensions is greater than the number of samples, uses a subset of training points, and are versatile with different kernel functions.

3. What is a potential issue when the number of features is much greater than the number of samples in support vector machines?
If the number of features is much greater than the number of samples, avoiding over-fitting in choosing kernel functions and regularization term is crucial.

4. Do support vector machines directly provide probability estimates?
No, support vector machines do not directly provide probability estimates. These are calculated using an expensive five-fold cross-validation.

5. What types of sample vectors do support vector machines in scikit-learn support?
Support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input.

In [42]:
from IPython.display import display, Markdown

In [38]:
chain.get_graph().print_ascii()

          +---------------------------+            
          | Parallel<notes,quiz>Input |            
          +---------------------------+            
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
    +----------+                  +----------+     
    | ChatGroq |                  | ChatGroq |     
    +----------+                  +----------+     
          *                             *          
          *                             *          
          *                             *          
+-----------------+            +-----------------+ 
| StrOutputP